# Faz 3 — YOLO CT Viewer

İnteraktif DICOM viewer: GT kutuları (mavi) + YOLO tahminleri (kırmızı).

In [1]:
!pip install ipywidgets

In [2]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

DATA_ROOT = Path(os.environ.get("TR_ABDOMEN_BASE", r"D:/makale-pdf/Proje/abdomen"))
PROJE     = Path(os.environ.get("TR_ABDOMEN_PROJE", r"D:/makale-pdf/Proje"))
CODE      = Path(os.environ.get("TR_ABDOMEN_CODE",  r"D:/makale-pdf/Proje/code"))
EGITIM_DIR = Path(os.environ.get("ABDOMEN_TRAIN_DIR", DATA_ROOT / "Egitim Verisi"))
YARISMA_DIR = Path(os.environ.get("ABDOMEN_TEST_DIR", DATA_ROOT / "Test Verisi"))

# os.environ["ABDOMEN_PROJECT_ROOT"] = str(PROJE)
# os.environ["ABDOMEN_DATA_ROOT"]    = str(DATA_ROOT)
# os.environ["ABDOMEN_TRAIN_DIR"]    = str(EGITIM_DIR)
# os.environ["ABDOMEN_TEST_DIR"]     = str(DATA_ROOT / "Yarışma Veri Seti")
# os.environ["ABDOMEN_BILGI_XLSX"]   = str(DATA_ROOT / "Bilgi.xlsx")
# os.environ["ABDOMEN_OUT_DIR"]      = str(PROJE / "outputs")

sys.path.insert(0, str(CODE))
from src.config import DET_DATA_DIR, CKPT_DIR, SUPER_CLASSES, DEFAULT_DET

fold = 0
fold_dir = DET_DATA_DIR / f"fold{fold}"
dataset_yaml = fold_dir / "dataset.yaml"
print("YOLO veri seti:", fold_dir, "→ var?", fold_dir.exists())
print("dataset.yaml :", dataset_yaml.exists())

YOLO veri seti: /Users/ramazanpolat/Desktop/datasets/abdomen/outputs/det_data/fold0 → var? True
dataset.yaml : True


In [3]:
import warnings
import threading
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image
from ipywidgets import IntSlider, Dropdown, HBox, VBox, Output, ToggleButton
from IPython.display import display
import pandas as pd
from src.splits import load_fold
from src.config import SUPER_CLASSES; warnings.filterwarnings("ignore")

CLASS_COLORS = [
    "#e74c3c", "#3498db", "#2ecc71", "#f39c12", "#9b59b6", "#1abc9c",
]

# ── Tahmin CSV ───────────────────────────────────────────────────────────────
_pred_csv = PROJE / "outputs" / "logs" / f"yolo_fold{fold}_val_preds.csv"
PRED_DF   = pd.read_csv(_pred_csv) if _pred_csv.exists() else pd.DataFrame()
print(f"Tahmin CSV : {_pred_csv}  ({'✓' if not PRED_DF.empty else '✗ yok — yalnızca GT gösterilir'})")

# ── Vaka → sınıf seti ───────────────────────────────────────────────────────
_fold_train = set(load_fold(fold, "train"))
_fold_val   = set(load_fold(fold, "val"))
_all_cases  = sorted(_fold_train | _fold_val)
_lbl_dir    = fold_dir / "labels"

def _case_classes_from_labels(case):
    classes = set()
    for split in ("train", "val"):
        for f in (_lbl_dir / split).glob(f"{case}_*.txt"):
            for line in f.read_text().strip().splitlines():
                parts = line.split()
                if parts:
                    try: classes.add(int(parts[0]))
                    except: pass
    return classes

_case_cls_map = {c: _case_classes_from_labels(c) for c in _all_cases}

# ── PNG listesi — DICOM yerine export edilmiş PNG kullanılır ─────────────────
# PNG adı {case}_{image_id}.png olduğundan label {case}_{image_id}.txt ile
# direkt eşleşir; DICOM yolu / kök dizin sorunları tamamen ortadan kalkar.
_png_cache: dict = {}

def _load_png_list(case):
    if case not in _png_cache:
        pngs = []
        for split in ("train", "val"):
            pngs.extend((fold_dir / "images" / split).glob(f"{case}_*.png"))
        _png_cache[case] = sorted(pngs, key=lambda p: int(p.stem.split("_", 1)[1]))
    return _png_cache[case]

# ── GT kutularını label .txt'den yükle ──────────────────────────────────────
def _load_gt(case, split):
    ldir = _lbl_dir / split
    rows = []
    for f in ldir.glob(f"{case}_*.txt"):
        parts = f.stem.split("_", 1)
        if len(parts) < 2: continue
        try: img_id = int(parts[1])
        except: continue
        for line in f.read_text().strip().splitlines():
            p = line.split()
            if len(p) < 5: continue
            cls = int(p[0]); cx, cy, w, h = map(float, p[1:5])
            rows.append({"image_id": img_id, "class": cls,
                         "cx": cx, "cy": cy, "w": w, "h": h})
    return pd.DataFrame(rows)


def yolo_viewer(fold=0):
    train_cases = set(load_fold(fold, "train"))
    val_cases   = set(load_fold(fold, "val"))
    all_cases   = sorted(train_cases | val_cases)

    split_dd = Dropdown(description="Split:", value="Tümü",
                        options=["Tümü", "Train", "Val"], layout={"width": "160px"})
    class_dd = Dropdown(description="Sınıf:", value="Tümü",
                        options=["Tümü"] + list(SUPER_CLASSES), layout={"width": "280px"})
    case_dd  = Dropdown(description="Vaka:", layout={"width": "180px"})
    slider   = IntSlider(description="Kesit:", min=0, max=1, value=0,
                         continuous_update=False, layout={"width": "500px"})
    gt_btn   = ToggleButton(value=True,  description="GT (Mavi)",
                            button_style="info",   layout={"width": "130px"})
    pred_btn = ToggleButton(value=True,  description="Tahmin (Kırmızı)",
                            button_style="danger", layout={"width": "160px"},
                            disabled=PRED_DF.empty)
    info_out = Output()
    img_out  = Output()

    def filtered_cases():
        sp = split_dd.value; cn = class_dd.value
        cases = all_cases
        if sp == "Train": cases = [c for c in cases if c in train_cases]
        elif sp == "Val": cases = [c for c in cases if c in val_cases]
        if cn != "Tümü":
            ci = SUPER_CLASSES.index(cn)
            cases = [c for c in cases if ci in _case_cls_map.get(c, set())]
        return cases

    def update_cases(_):
        cases = filtered_cases()
        case_dd.options = cases
        if cases: case_dd.value = cases[0]

    def draw(z_idx, case, show_gt, show_pred):
        pngs = _load_png_list(case)
        if not pngs or z_idx >= len(pngs):
            with img_out:
                img_out.clear_output(wait=True)
                print(f"⚠️  Vaka {case}: export edilmiş PNG bulunamadı.")
            return

        png_path = pngs[z_idx]
        img_id   = int(png_path.stem.split("_", 1)[1])
        split    = png_path.parent.name  # "train" veya "val"

        img = np.array(Image.open(png_path)).astype(np.float32) / 255.0
        H, W = img.shape[:2]

        gt_df = _load_gt(case, split)

        n_cols = 1 + int(show_gt) + int(show_pred and not PRED_DF.empty)
        fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 5), facecolor="#111")
        if n_cols == 1: axes = [axes]
        col = 0

        # Ham CT
        axes[col].imshow(img, vmin=0, vmax=1)
        axes[col].set_title(f"CT  z={z_idx+1}/{len(pngs)}  img_id={img_id}",
                            color="white", fontsize=9)
        axes[col].axis("off"); col += 1

        # GT kutuları (mavi)
        if show_gt:
            axes[col].imshow(img, vmin=0, vmax=1)
            sl_gt = gt_df[gt_df["image_id"] == img_id] if not gt_df.empty else pd.DataFrame()
            for _, r in sl_gt.iterrows():
                cls = int(r["class"])
                x1 = (r.cx - r.w / 2) * W;  y1 = (r.cy - r.h / 2) * H
                bw  = r.w * W;               bh = r.h * H
                axes[col].add_patch(mpatches.Rectangle(
                    (x1, y1), bw, bh,
                    linewidth=2, edgecolor="#3498db", facecolor="none"))
                lbl = SUPER_CLASSES[cls] if cls < len(SUPER_CLASSES) else str(cls)
                axes[col].text(x1, max(y1 - 3, 0), lbl,
                               color="#3498db", fontsize=6, weight="bold",
                               bbox=dict(facecolor="black", alpha=0.4, pad=1))
            axes[col].set_title(f"GT ({len(sl_gt)} kutu)  img_id={img_id}",
                                color="white", fontsize=9)
            axes[col].axis("off"); col += 1

        # Tahmin kutuları (kırmızı)
        if show_pred and not PRED_DF.empty:
            axes[col].imshow(img, vmin=0, vmax=1)
            sl_pred = PRED_DF[(PRED_DF["case"] == case) & (PRED_DF["image_id"] == img_id)]
            for _, r in sl_pred.iterrows():
                cls   = int(r["class"])
                color = CLASS_COLORS[cls % len(CLASS_COLORS)]
                axes[col].add_patch(mpatches.Rectangle(
                    (r.x1, r.y1), r.x2 - r.x1, r.y2 - r.y1,
                    linewidth=2, edgecolor=color, facecolor="none"))
                lbl = SUPER_CLASSES[cls] if cls < len(SUPER_CLASSES) else str(cls)
                axes[col].text(r.x1, max(r.y1 - 3, 0),
                               f"{lbl[:12]} {r.score:.2f}",
                               color=color, fontsize=6, weight="bold",
                               bbox=dict(facecolor="black", alpha=0.4, pad=1))
            axes[col].set_title(f"Tahmin ({len(sl_pred)} kutu)  img_id={img_id}",
                                color="white", fontsize=9)
            axes[col].axis("off")

        patches = [mpatches.Patch(color=CLASS_COLORS[i], label=SUPER_CLASSES[i])
                   for i in range(len(SUPER_CLASSES))]
        fig.legend(handles=patches, loc="lower center", ncol=3,
                   fontsize=7, facecolor="#222", labelcolor="white", framealpha=0.8)
        tag = "Train" if case in train_cases else "Val"
        fig.suptitle(
            f"Vaka {case}  [{tag}]  |  "
            f"Sınıflar: {', '.join(SUPER_CLASSES[c] for c in sorted(_case_cls_map.get(case, set())))}",
            color="white", fontsize=10, fontweight="bold")
        plt.tight_layout(rect=[0, 0.08, 1, 1])
        with img_out:
            img_out.clear_output(wait=True); plt.show()

    def on_case_change(_):
        case = case_dd.value
        if case is None: return
        pngs = _load_png_list(case)
        n    = len(pngs)
        slider.max   = max(0, n - 1)
        slider.value = n // 2
        with info_out:
            info_out.clear_output()
            tag   = "Train" if case in train_cases else "Val"
            gt_df = _load_gt(case, tag.lower())
            ann   = gt_df["image_id"].nunique() if not gt_df.empty else 0
            n_box = len(gt_df) if not gt_df.empty else 0
            print(f"Vaka {case}  [{tag}]  |  {n} kesit  |  "
                  f"Annotasyonlu: {ann} kesit  |  {n_box} GT kutu  |  "
                  f"Sınıflar: {', '.join(SUPER_CLASSES[c] for c in sorted(_case_cls_map.get(case, set())))}")
        draw(slider.value, case, gt_btn.value, pred_btn.value)

    def on_change(_):
        if case_dd.value is not None:
            draw(slider.value, case_dd.value, gt_btn.value, pred_btn.value)

    for w in [split_dd, class_dd]: w.observe(update_cases, names="value")
    case_dd.observe(on_case_change, names="value")
    for w in [slider, gt_btn, pred_btn]: w.observe(on_change, names="value")

    display(VBox([
        HBox([split_dd, class_dd]),
        HBox([case_dd, gt_btn, pred_btn]),
        info_out, slider, img_out,
    ]))
    threading.Timer(0.3, update_cases, args=[None]).start()


Tahmin CSV : /Users/ramazanpolat/Desktop/datasets/abdomen/outputs/logs/yolo_fold0_val_preds.csv  (✗ yok — yalnızca GT gösterilir)


In [ ]:
yolo_viewer(fold=fold)